# SIC 2026 · Inteligencia Artificial
## Clase 10: Ejercicio final de aprendizaje no supervisado e introducción al procesamiento de lenguaje natural

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

Hoy cerramos el Capítulo 6 con un ejercicio completo y abrimos el Capítulo 7.

| Hora | Parte | Qué vamos a hacer |
|---|---|---|
| 18:00 a 19:10 | 1. Ejercicio final no supervisado | Agrupar 167 países según indicadores de desarrollo y decidir cuáles priorizar para ayuda |
| 19:10 a 19:20 | Pausa | |
| 19:20 a 19:40 | 2. ¿Qué es el PLN? | Por qué el texto es un tipo de dato distinto |
| 19:40 a 20:20 | 3. Obtener y limpiar texto | Traer texto desde una página web y limpiarlo con expresiones regulares |
| 20:20 a 21:10 | 4. Preprocesamiento | Tokenizar, quitar stopwords, stemming y lematización |
| 21:10 a 21:50 | 5. Del texto a números | Bolsa de palabras y TF-IDF |
| 21:50 a 22:00 | 6. Puente | Agrupar noticias con TF-IDF y K-medias |

## Cómo está organizado este notebook

Cada paso tiene siempre la misma estructura:

- **Qué vamos a hacer:** el objetivo del paso en una o dos frases.
- **El código:** ejecútalo con `Shift + Enter`.
- **Qué deberías ver:** el resultado esperado, para que compruebes que todo va bien.

Los ejercicios tienen una celda de verificación que responde "Correcto." o "Todavía no.". Las soluciones están al final.

**Antes de empezar:** *Archivo > Guardar una copia en Drive*.

---
# Paso 0. Preparar el entorno

**Qué vamos a hacer:** instalar el modelo de español de spaCy (lo usaremos para lematizar en la Parte 4), cargar las librerías y definir la función `revisar`.

Esta celda tarda alrededor de un minuto la primera vez.

In [ ]:
!pip install -q kagglehub
!python -m spacy download es_core_news_sm -q

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import nltk
nltk.download("stopwords", quiet=True)

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    """Imprime si un ejercicio está bien resuelto."""
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

print("Listo.")

**Qué deberías ver:** la palabra `Listo.` al final (pueden aparecer mensajes de instalación antes).

---
# Parte 1. Ejercicio final de aprendizaje no supervisado

## El caso

Una ONG internacional de ayuda humanitaria reunió fondos para apoyar a los países que más lo necesitan. El directorio tiene que decidir **dónde concentrar la ayuda**, y le pide al equipo de datos una recomendación basada en indicadores socioeconómicos y de salud.

No hay una "respuesta correcta" en los datos: nadie etiquetó a los países como "necesita ayuda" o "no necesita ayuda". Por eso es un problema **no supervisado**: vamos a buscar grupos de países parecidos y luego interpretarlos.

Usaremos todo lo que vimos en las Clases 7 y 8: escalado, PCA, K-medias con silueta, clustering jerárquico y DBSCAN.

## Paso 1.1. Cargar los datos

**Qué vamos a hacer:** descargar desde Kaggle el dataset *Unsupervised Learning on Country Data*. Si Kaggle no responde, la función usa una copia pública del mismo archivo.

In [ ]:
def cargar_paises():
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("rohan0301/unsupervised-learning-on-country-data")
        print("Descargado desde Kaggle.")
        return pd.read_csv(os.path.join(carpeta, "Country-data.csv"))
    except Exception:
        print("Kaggle no respondió. Uso la copia de respaldo.")
        return pd.read_csv("https://raw.githubusercontent.com/Ysnyldrms/Unsupervised-Learning-on-Country-Data/main/Country-data.csv")

paises = cargar_paises()
print("Forma:", paises.shape)
paises.head()

**Qué deberías ver:** `Forma: (167, 10)`: 167 países y 10 columnas.

| Columna | Significado |
|---|---|
| `country` | Nombre del país (en inglés) |
| `child_mort` | Muertes de niños menores de 5 años por cada 1.000 nacidos vivos |
| `exports` | Exportaciones, como porcentaje del PIB |
| `health` | Gasto total en salud, como porcentaje del PIB |
| `imports` | Importaciones, como porcentaje del PIB |
| `income` | Ingreso neto por persona (dólares, ajustado por poder de compra) |
| `inflation` | Inflación anual (%) |
| `life_expec` | Esperanza de vida al nacer (años) |
| `total_fer` | Hijos por mujer |
| `gdpp` | PIB por persona (dólares) |

Los datos corresponden a un año fijo, alrededor de 2010. Sirven para practicar, no como diagnóstico actual de ningún país.

## Paso 1.2. Mirar los datos antes de modelar

**Qué vamos a hacer:** revisar que no falten datos, ver las estadísticas y ubicar a Chile.

In [ ]:
print("Valores faltantes:", paises.isna().sum().sum())
paises.describe().round(1)

In [ ]:
paises[paises["country"].isin(["Chile", "Argentina", "Peru", "Uruguay", "Bolivia"])]

**Qué deberías ver:** cero faltantes y una tabla donde las columnas tienen **escalas muy distintas**: `income` llega a 125.000, mientras `total_fer` no pasa de 8.

**Para pensar:** si no escalamos, ¿qué variable dominaría las distancias? (Repaso de la Clase 8: `income` y `gdpp`, solo por estar en dólares.)

### Ejercicio 1
Guarda en `esperanza_chile` la esperanza de vida (`life_expec`) de Chile, como número.

In [ ]:
esperanza_chile = None  # TU CÓDIGO AQUÍ
print(esperanza_chile)

In [ ]:
revisar(lambda: np.isclose(esperanza_chile, 79.1))

## Paso 1.3. Escalar

**Qué vamos a hacer:** separar los nombres de los países de los números, y escalar los números para que todas las variables pesen parecido.

In [ ]:
from sklearn.preprocessing import StandardScaler

nombres = paises["country"]
indicadores = paises.drop(columns="country")
X = StandardScaler().fit_transform(indicadores)

print("Forma de X:", X.shape)
print("Promedios después de escalar (deberían ser casi 0):", X.mean(axis=0).round(2))

**Qué deberías ver:** `Forma de X: (167, 9)` y una lista de ceros.

## Paso 1.4. PCA: ver a los 167 países en un mapa

**Qué vamos a hacer:** resumir los 9 indicadores en componentes, ver cuánta información guarda cada una y ponerles nombre mirando sus cargas.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA().fit(X)
var = pca.explained_variance_ratio_
print("Varianza explicada por componente:", var.round(3))
print("Varianza acumulada:              ", var.cumsum().round(3))

cargas = pd.DataFrame(pca.components_[:2].T, index=indicadores.columns, columns=["Componente 1", "Componente 2"])
cargas.round(2).sort_values("Componente 1")

**Qué deberías ver:** la primera componente explica cerca del 46% y las dos primeras, cerca del 63%.

**Cómo nombrar las componentes** (recuerden que el signo puede aparecer invertido):

- **Componente 1:** mortalidad infantil y fecundidad van hacia un lado; ingreso, PIB y esperanza de vida hacia el otro. Es un eje de **desarrollo**.
- **Componente 2:** dominada por exportaciones e importaciones. Es un eje de **apertura comercial**.

### Ejercicio 2
¿Cuántas componentes se necesitan para explicar **al menos el 80%** de la variación? Guárdalo en `n_comp_80`. (Pista: Clase 8, `np.argmax`.)

In [ ]:
n_comp_80 = None  # TU CÓDIGO AQUÍ
print(n_comp_80)

In [ ]:
revisar(lambda: n_comp_80 == int(np.argmax(var.cumsum() >= 0.8) + 1))

Ahora el mapa. Marcamos algunos países para orientarnos.

In [ ]:
Z = PCA(n_components=2).fit_transform(X)
mapa = pd.DataFrame({"pais": nombres, "C1": Z[:, 0], "C2": Z[:, 1]})

DESTACADOS = ["Chile", "Haiti", "Luxembourg", "Qatar", "Nigeria", "Singapore", "Norway", "Bolivia", "India", "United States"]

plt.figure(figsize=(10, 6))
plt.scatter(mapa["C1"], mapa["C2"], s=18, alpha=0.5)
for _, f in mapa[mapa["pais"].isin(DESTACADOS)].iterrows():
    plt.annotate(f["pais"], (f["C1"], f["C2"]), fontsize=9, xytext=(4, 4), textcoords="offset points")
plt.xlabel("Componente 1 (desarrollo)"); plt.ylabel("Componente 2 (apertura comercial)")
plt.title("167 países resumidos en 2 componentes"); plt.grid(alpha=0.3); plt.show()

**Qué deberías ver:** países como Haití en un extremo del eje de desarrollo y países como Noruega en el otro. Luxemburgo y Singapur quedan muy arriba: exportan e importan más que todo su PIB, porque son centros de comercio.

## Paso 1.5. K-medias: ¿cuántos grupos?

**Qué vamos a hacer:** probar k de 2 a 7, mirar la silueta y el tamaño de cada grupo.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

for k in range(2, 8):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    print(f"k = {k} | silueta = {silhouette_score(X, etiquetas):.3f} | tamaños = {np.bincount(etiquetas)}")

**Qué deberías ver:** siluetas parecidas entre k = 2 y k = 5 (cerca de 0,29), así que la silueta no decide sola.

Miren los **tamaños**: con k = 4 aparece un grupo de solo 3 países, y con k = 5, uno de 1. Esos grupos minúsculos no son un "tipo de país": son **casos atípicos** que K-medias aisló. Los veremos en el Paso 1.8 con DBSCAN.

**Decisión:** usamos **k = 3**. Tiene silueta parecida a las demás, grupos de tamaño razonable y, como veremos, una interpretación clara para la ONG.

## Paso 1.6. Interpretar los grupos

**Qué vamos a hacer:** entrenar K-medias con k = 3, calcular el promedio de cada indicador por grupo y ponerles nombre.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
paises["grupo"] = km.labels_

perfil = paises.groupby("grupo").agg(
    paises=("country", "size"),
    mortalidad_infantil=("child_mort", "mean"),
    ingreso=("income", "mean"),
    esperanza_vida=("life_expec", "mean"),
    hijos_por_mujer=("total_fer", "mean"),
    pib_persona=("gdpp", "mean"),
    inflacion=("inflation", "mean"),
).round(1).sort_values("mortalidad_infantil", ascending=False)
perfil

**Qué deberías ver:** tres grupos muy distintos (los números de grupo pueden variar; guíense por los valores):

- Un grupo de unos 47 países con **mortalidad infantil cercana a 93 por mil**, ingreso promedio cercano a 4.000 dólares y esperanza de vida de unos 59 años.
- Un grupo de unos 84 países en un **nivel intermedio**.
- Un grupo de unos 36 países con **mortalidad infantil cercana a 5 por mil** e ingreso cercano a 46.000 dólares.

Les damos nombre y vemos dónde quedó Chile:

In [ ]:
orden = perfil.index.tolist()          # de mayor a menor mortalidad infantil
NOMBRES_GRUPOS = {orden[0]: "Necesita ayuda", orden[1]: "En desarrollo", orden[2]: "Desarrollo alto"}
paises["grupo_nombre"] = paises["grupo"].map(NOMBRES_GRUPOS)

print(paises["grupo_nombre"].value_counts())
print("\nChile quedó en el grupo:", paises.loc[paises["country"] == "Chile", "grupo_nombre"].item())

**Qué deberías ver:** Chile en el grupo "En desarrollo", junto con la mayoría de los países de América Latina.

### Ejercicio 3
¿Cuántos países quedaron en **el mismo grupo que Chile**? Guárdalo en `n_grupo_chile` (incluye a Chile).

In [ ]:
n_grupo_chile = None  # TU CÓDIGO AQUÍ
print(n_grupo_chile)

In [ ]:
revisar(lambda: n_grupo_chile == (paises["grupo"] == paises.loc[paises["country"] == "Chile", "grupo"].item()).sum())

Veamos los grupos en el mapa de PCA:

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x=Z[:, 0], y=Z[:, 1], hue=paises["grupo_nombre"], palette={"Necesita ayuda": "#D9534F", "En desarrollo": "#E8A317", "Desarrollo alto": "#1E9E73"}, s=35)
for _, f in mapa[mapa["pais"].isin(DESTACADOS)].iterrows():
    plt.annotate(f["pais"], (f["C1"], f["C2"]), fontsize=9, xytext=(4, 4), textcoords="offset points")
plt.xlabel("Componente 1 (desarrollo)"); plt.ylabel("Componente 2 (apertura comercial)")
plt.title("Tres grupos de países encontrados por K-medias"); plt.legend(title=""); plt.show()

## Paso 1.7. Comparar con el clustering jerárquico

**Qué vamos a hacer:** ver si un método distinto llega a grupos parecidos. Si dos métodos coinciden, confiamos más en el resultado.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.cluster import AgglomerativeClustering

plt.figure(figsize=(12, 4.5))
dendrogram(linkage(X, method="ward"), truncate_mode="lastp", p=25, leaf_font_size=9)
plt.title("Dendrograma (Ward), mostrando solo las últimas 25 uniones"); plt.ylabel("Distancia de unión"); plt.show()

jerarquico = AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X)
pd.crosstab(paises["grupo_nombre"], jerarquico, rownames=["K-medias"], colnames=["Jerárquico"])

**Qué deberías ver:** con 167 países, el dendrograma completo sería ilegible, por eso `truncate_mode="lastp"` muestra solo las últimas uniones (los números entre paréntesis indican cuántos países hay en cada rama). En la tabla, los métodos coinciden casi por completo en "Desarrollo alto" y "En desarrollo". La diferencia está en "Necesita ayuda": el jerárquico es más estricto, deja ahí a unos 27 de los 47 países y pasa unos 20 al grupo intermedio.

**Mensaje:** el núcleo de cada grupo es estable, pero los países de la frontera cambian según el método. Esos 20 países son justamente los casos difíciles para la ONG: una recomendación seria debe revisarlos uno a uno.

## Paso 1.8. DBSCAN: ¿qué países son atípicos?

**Qué vamos a hacer:** usar el gráfico de k-distancia para elegir eps y encontrar países que no se parecen a ningún grupo.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

distancias, _ = NearestNeighbors(n_neighbors=5).fit(X).kneighbors(X)
plt.figure(figsize=(7, 4))
plt.plot(np.sort(distancias[:, -1])); plt.axhline(2.5, ls="--", color="gray", label="eps = 2,5")
plt.xlabel("Países ordenados"); plt.ylabel("Distancia al 5.º vecino"); plt.title("Gráfico de k-distancia"); plt.legend(); plt.show()

etiquetas_db = DBSCAN(eps=2.5, min_samples=5).fit_predict(X)
atipicos = paises.loc[etiquetas_db == -1, ["country", "child_mort", "income", "inflation", "life_expec", "exports", "gdpp"]]
atipicos

**Qué deberías ver:** la curva sube bruscamente cerca de 2,5, y DBSCAN marca 6 países como atípicos. Cada uno lo es por una razón distinta:

| País | Qué tiene de extremo |
|---|---|
| Haití | La mortalidad infantil más alta (208 por mil) y una esperanza de vida de 32 años |
| Nigeria | Inflación de 104% anual |
| Qatar | El ingreso más alto del dataset (125.000 dólares) |
| Luxemburgo, Malta y Singapur | Exportaciones e importaciones muy por encima de su PIB |

**Para la ONG esto es clave:** Haití y Nigeria necesitan un análisis individual, porque un promedio de grupo no los representa bien.

### Ejercicio 4
¿Cuántos países marca DBSCAN como atípicos con **eps = 2.0** (y `min_samples=5`)? Guárdalo en `n_atipicos_2`. ¿Por qué cambia?

In [ ]:
n_atipicos_2 = None  # TU CÓDIGO AQUÍ
print(n_atipicos_2)

In [ ]:
revisar(lambda: n_atipicos_2 == (DBSCAN(eps=2.0, min_samples=5).fit_predict(X) == -1).sum())

## Paso 1.9. La recomendación

**Qué vamos a hacer:** dentro del grupo "Necesita ayuda", ordenar a los países por mortalidad infantil para proponer una lista de prioridad.

In [ ]:
prioridad = (paises[paises["grupo_nombre"] == "Necesita ayuda"]
             .sort_values("child_mort", ascending=False)
             [["country", "child_mort", "income", "life_expec", "health"]]
             .head(10).reset_index(drop=True))
prioridad.index += 1
prioridad

**Qué deberías ver:** Haití, Sierra Leona, Chad y República Centroafricana entre los primeros.

**Cómo presentaríamos la recomendación al directorio:**

1. Los 167 países forman tres grupos claros. El grupo que necesita ayuda tiene unos 47 países, con mortalidad infantil cercana a 93 por mil.
2. Dentro de ese grupo, proponemos priorizar según la mortalidad infantil, que es el indicador más directo de vulnerabilidad.
3. Haití y Nigeria son casos atípicos: requieren un análisis propio antes de decidir.
4. **Límite importante:** los datos son de un año fijo y no consideran guerras, desastres recientes ni capacidad local para ejecutar la ayuda. El modelo ordena información; la decisión es humana.

### Ejercicio 5
La mortalidad infantil no es el único criterio posible. Ordena el grupo "Necesita ayuda" por **ingreso de menor a mayor** y guarda en `top5_ingreso` una **lista** con los nombres de los 5 primeros países.

In [ ]:
top5_ingreso = None  # TU CÓDIGO AQUÍ
print(top5_ingreso)

In [ ]:
_ref = paises[paises["grupo_nombre"] == "Necesita ayuda"].sort_values("income")["country"].head(5).tolist()
revisar(lambda: list(top5_ingreso) == _ref)

**Para pensar:** ¿la lista cambia mucho según el criterio? ¿Qué criterio usarían ustedes y por qué? Esta es la parte del trabajo de datos que no hace el algoritmo.

---
# Parte 2. ¿Qué es el procesamiento de lenguaje natural?

> **Definición. Procesamiento de lenguaje natural (PLN):** el área de la IA que permite a los computadores trabajar con el lenguaje humano: entenderlo, clasificarlo, resumirlo, traducirlo o generarlo.

Hasta ahora todos nuestros datos eran **tablas**: cada columna era un número o una categoría. El texto es distinto:

| | Datos en tabla | Texto |
|---|---|---|
| Estructura | Filas y columnas fijas | Cada documento tiene un largo distinto |
| Unidad | Un número o una categoría | Palabras, que dependen del contexto |
| ¿Se puede usar directo en un modelo? | Sí, después de escalar | No: primero hay que convertirlo en números |

Tres palabras que usaremos todo el capítulo:

- **Documento:** cada texto que analizamos (un correo, una reseña, una noticia).
- **Corpus:** el conjunto de documentos.
- **Token:** cada unidad en que dividimos el texto, normalmente una palabra.

**Tareas típicas de PLN:**

| Tarea | Ejemplo | Capítulo |
|---|---|---|
| Clasificar textos | ¿Este correo es spam? ¿Esta reseña es positiva? | 7 |
| Agrupar textos | ¿De qué temas hablan estas 10.000 respuestas de una encuesta? | 7 (hoy) |
| Extraer información | Encontrar fechas, montos y nombres en contratos | 7 (hoy) |
| Generar texto | Resumir, traducir, conversar | 10 |

El flujo de hoy es el que se usa en casi todo proyecto de PLN:

**Obtener texto → limpiarlo → preprocesarlo → convertirlo en números → aplicar un modelo**

---
# Parte 3. Obtener y limpiar texto

## Paso 3.1. Traer texto desde una página web (web scraping)

> **Definición. Web scraping:** extraer información de páginas web con código.

Usamos dos librerías:

- **Requests:** descarga el código HTML de la página, igual que lo hace el navegador.
- **BeautifulSoup:** lee ese HTML y permite buscar partes específicas, como los párrafos (`<p>`).

**Qué vamos a hacer:** descargar el artículo de Wikipedia en español sobre procesamiento de lenguaje natural y quedarnos solo con los párrafos. Si la red falla, usamos un texto de respaldo.

**Buenas prácticas:** revisen los términos de uso del sitio, no hagan cientos de peticiones seguidas y nunca extraigan datos personales.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL = "https://es.wikipedia.org/wiki/Procesamiento_de_lenguajes_naturales"
TEXTO_RESPALDO = (
    "El procesamiento de lenguajes naturales es un campo de las ciencias de la computación, de la inteligencia "
    "artificial y de la lingüística que estudia las interacciones entre las computadoras y el lenguaje humano [1]. "
    "Se ocupa de la formulación e investigación de mecanismos eficaces computacionalmente para la comunicación entre "
    "personas y máquinas por medio del lenguaje natural [2]. Algunas aplicaciones son la traducción automática, "
    "la clasificación de documentos, el análisis de sentimientos y los asistentes conversacionales [3]."
)

try:
    respuesta = requests.get(URL, headers={"User-Agent": "Mozilla/5.0 (curso SIC 2026)"}, timeout=10)
    respuesta.raise_for_status()
    sopa = BeautifulSoup(respuesta.text, "html.parser")
    parrafos = [p.get_text() for p in sopa.find_all("p") if len(p.get_text().strip()) > 80]
    texto_web = " ".join(parrafos[:5])
    print("Descargado desde Wikipedia:", len(parrafos), "párrafos largos encontrados")
except Exception as e:
    texto_web = TEXTO_RESPALDO
    print("No se pudo descargar la página. Uso el texto de respaldo.")

print("\n", texto_web[:600])

**Qué deberías ver:** los primeros 600 caracteres del artículo. Fíjense en que el texto trae "ruido": números entre corchetes como `[1]`, que son referencias de Wikipedia y no nos sirven.

**Cómo funciona `find_all("p")`:** en HTML, cada párrafo está entre las etiquetas `<p>` y `</p>`. BeautifulSoup encuentra todas esas etiquetas y `get_text()` saca el texto sin las etiquetas.

## Paso 3.2. Expresiones regulares

> **Definición. Expresión regular (regex):** un patrón que describe un tipo de texto. Sirve para **buscar**, **extraer** o **reemplazar** partes de un texto que siguen una forma, aunque no sepamos exactamente qué dicen.

Los símbolos más usados:

| Símbolo | Significa | Ejemplo | Encuentra |
|---|---|---|---|
| `\d` | Un dígito | `\d\d` | `42` |
| `\w` | Una letra, dígito o guion bajo | `\w+` | `hola` |
| `\s` | Un espacio | `a\sb` | `a b` |
| `.` | Cualquier carácter | `c.sa` | `casa`, `cosa` |
| `+` | Una o más veces | `\d+` | `7`, `2026` |
| `*` | Cero o más veces | `ab*` | `a`, `abbb` |
| `?` | Opcional | `colou?r` | `color`, `colour` |
| `{n}` / `{n,m}` | Exactamente n / entre n y m veces | `\d{4}` | `2026` |
| `[...]` | Uno de estos caracteres | `[kK]` | `k` o `K` |
| `\` | Usa el siguiente símbolo literalmente | `\.` | un punto |

En Python usamos el módulo `re`:

- `re.findall(patron, texto)`: devuelve una lista con todo lo que coincide.
- `re.sub(patron, reemplazo, texto)`: reemplaza lo que coincide.

**Qué vamos a hacer:** quitar las referencias `[1]`, `[2]` del texto de Wikipedia.

In [ ]:
texto_limpio = re.sub(r"\[\d+\]", "", texto_web)
print("Antes:  ", texto_web[:200])
print("Después:", texto_limpio[:200])

**Cómo leer el patrón `\[\d+\]`:** un corchete literal `\[`, uno o más dígitos `\d+`, y otro corchete literal `\]`. Los corchetes llevan `\` porque, sin ella, `[...]` significa "uno de estos caracteres".

**Qué vamos a hacer ahora:** extraer datos con forma conocida de un texto de ejemplo: correos, montos en pesos, fechas, RUT y teléfonos.

In [ ]:
mensaje = ("Escríbenos a contacto@innovacien.cl o a ventas.chile@ejemplo.com. El curso cuesta $129.990 y la matrícula "
           "$25.000. Inscripciones hasta el 15/11/2026. Para la boleta, indica tu RUT, por ejemplo 12.345.678-5 "
           "o 9.876.543-K. Llama al +56 9 8765 4321.")

patrones = {
    "Correos":   r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "Montos":    r"\$\d{1,3}(?:\.\d{3})*",
    "Fechas":    r"\b\d{2}/\d{2}/\d{4}\b",
    "RUT":       r"\b\d{1,2}\.\d{3}\.\d{3}-[\dkK]\b",
    "Teléfonos": r"\+56 9 \d{4} \d{4}",
}
for nombre, patron in patrones.items():
    print(f"{nombre:10s} {re.findall(patron, mensaje)}")

**Qué deberías ver:** 2 correos, 2 montos, 1 fecha, 2 RUT y 1 teléfono.

**Cómo leer el patrón del RUT** `\d{1,2}\.\d{3}\.\d{3}-[\dkK]`: uno o dos dígitos, un punto, tres dígitos, un punto, tres dígitos, un guion, y un dígito verificador que puede ser número o K.

`(?:...)` agrupa una parte del patrón para repetirla sin guardarla aparte. En los montos, `(?:\.\d{3})*` significa "cero o más grupos de punto y tres dígitos".

### Ejercicio 6
Escribe un patrón `patron_porcentaje` que encuentre porcentajes como `45%`, `3,5%` y `100%` en el texto de la celda. Pista: dígitos, luego opcionalmente una coma con más dígitos, y luego `%`.

In [ ]:
texto_porcentajes = "La aprobación subió a 45% y la inflación anual fue de 3,5%. La meta es llegar al 100% de cobertura."

patron_porcentaje = None  # TU CÓDIGO AQUÍ, por ejemplo r"..."
print(re.findall(patron_porcentaje, texto_porcentajes) if patron_porcentaje else None)

In [ ]:
revisar(lambda: re.findall(patron_porcentaje, texto_porcentajes) == ["45%", "3,5%", "100%"])

---
# Parte 4. Preprocesamiento de texto

Para que un modelo compare textos, conviene que "Economía", "economía" y "económica" no sean tres cosas distintas, y que palabras como "el", "de" o "que", que aparecen en todos los textos, no dominen el análisis. Eso es el **preprocesamiento**.

Lo haremos paso a paso con una sola frase, para ver qué hace cada paso.

In [ ]:
frase = "Los Bancos Centrales subieron las tasas de interés, pero la economía chilena creció 2,5% en 2026."
print(frase)

## Paso 4.1. Minúsculas

**Qué vamos a hacer:** pasar todo a minúsculas, para que "Bancos" y "bancos" sean la misma palabra.

In [ ]:
paso1 = frase.lower()
print(paso1)

## Paso 4.2. Quitar puntuación y números

**Qué vamos a hacer:** con una expresión regular, reemplazar por un espacio todo lo que **no** sea una letra. `[^...]` significa "cualquier carácter que no esté en la lista".

Ojo: en otros problemas los números sí importan (por ejemplo, montos en reclamos). Siempre hay que decidir según el objetivo.

In [ ]:
paso2 = re.sub(r"[^a-záéíóúñü\s]", " ", paso1)
print(paso2)

## Paso 4.3. Tokenizar

> **Definición. Tokenizar:** dividir el texto en unidades, normalmente palabras. Cada unidad es un **token**.

La forma más simple es separar por espacios con `split()`.

In [ ]:
tokens = paso2.split()
print(tokens)
print("Número de tokens:", len(tokens))

## Paso 4.4. Quitar stopwords

> **Definición. Stopwords (palabras vacías):** palabras muy frecuentes que casi no aportan significado sobre el tema, como artículos, preposiciones y conjunciones: "el", "la", "de", "que", "pero".

NLTK trae una lista de 313 stopwords en español.

In [ ]:
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words("spanish"))
print("Algunas stopwords:", sorted(STOPWORDS)[:20])

paso4 = [t for t in tokens if t not in STOPWORDS]
print("\nSin stopwords:", paso4)

**Qué deberías ver:** desaparecen "los", "las", "de", "pero", "la" y "en".

## Paso 4.5. Stemming y lematización

Las dos técnicas buscan que distintas formas de una palabra cuenten como una sola. La diferencia está en cómo lo hacen:

| | Stemming | Lematización |
|---|---|---|
| Qué hace | Corta la palabra para dejar su raíz | Busca la forma de diccionario |
| "subieron" | `sub` | `subir` |
| "chilena" | `chilen` | `chileno` |
| Resultado | A veces no es una palabra real | Siempre es una palabra real |
| Velocidad | Muy rápido | Más lento: necesita un modelo del idioma |
| Herramienta | NLTK `SnowballStemmer` | spaCy |

In [ ]:
from nltk.stem import SnowballStemmer
import spacy

stemmer = SnowballStemmer("spanish")
nlp = spacy.load("es_core_news_sm")

comparacion = pd.DataFrame({
    "palabra": paso4,
    "stemming": [stemmer.stem(t) for t in paso4],
    "lematización": [nlp(t)[0].lemma_ for t in paso4],
})
comparacion

**Qué deberías ver:** el stemming deja raíces como `econom` o `chilen`; la lematización deja palabras completas como `economía` o `chileno`. Para agrupar o clasificar textos, ambas sirven; el stemming es más rápido y la lematización es más fácil de leer.

**Nota:** aquí lematizamos palabra por palabra para comparar. En la práctica conviene pasarle a spaCy la frase completa, porque usa el contexto para decidir el lema.

## Paso 4.6. Todo junto en una función

**Qué vamos a hacer:** juntar los pasos en una función `preprocesar` que usaremos el resto de la clase.

In [ ]:
def preprocesar(texto, metodo="stemming"):
    texto = texto.lower()                                   # 1. minúsculas
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)          # 2. quitar puntuación y números
    tokens = [t for t in texto.split()                      # 3. tokenizar
              if t not in STOPWORDS and len(t) > 2]         # 4. quitar stopwords y palabras muy cortas
    if metodo == "stemming":                                # 5. normalizar
        tokens = [stemmer.stem(t) for t in tokens]
    elif metodo == "lematizacion":
        tokens = [t.lemma_ for t in nlp(" ".join(tokens))]
    return " ".join(tokens)

print("Original:     ", frase)
print("Stemming:     ", preprocesar(frase))
print("Lematización: ", preprocesar(frase, "lematizacion"))

### Ejercicio 7
Usa `preprocesar` (con stemming) sobre la frase de la celda y guarda el resultado en `frase_procesada`. ¿Cuántos tokens quedan?

In [ ]:
otra_frase = "Las jugadoras chilenas ganaron el campeonato y celebraron con la hinchada en el estadio."

frase_procesada = None  # TU CÓDIGO AQUÍ
print(frase_procesada)

In [ ]:
revisar(lambda: frase_procesada == preprocesar(otra_frase))

## Paso 4.7. N-gramas

> **Definición. N-grama:** una secuencia de n tokens seguidos. Los **bigramas** (n = 2) capturan expresiones como "inteligencia artificial" o "banco central", que pierden su sentido si se separan.

In [ ]:
def ngramas(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

tokens_ejemplo = "el banco central subió la tasa de interés".split()
print("Bigramas: ", ngramas(tokens_ejemplo, 2))
print("Trigramas:", ngramas(tokens_ejemplo, 3))

---
# Parte 5. Del texto a números

Los modelos trabajan con números. Hay que convertir cada documento en un vector.

## Paso 5.1. Bolsa de palabras

> **Definición. Bolsa de palabras (bag of words):** representa cada documento como el **conteo** de cada palabra del vocabulario, sin importar el orden. El resultado es una **matriz documento-término**: una fila por documento y una columna por palabra.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

mini_corpus = ["gol partido gol", "partido tenis", "tasa interés tasa"]

contador = CountVectorizer()
conteos = contador.fit_transform(mini_corpus)
pd.DataFrame(conteos.toarray(), columns=contador.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"])

**Qué deberías ver:** una tabla de 3 × 5. El documento 1 tiene `gol = 2` y `partido = 1`.

**El problema de contar:** una palabra que aparece en **todos** los documentos cuenta mucho, aunque no sirva para distinguirlos. Necesitamos premiar las palabras que son características de un documento.

## Paso 5.2. TF-IDF, calculado a mano

> **Definición. TF-IDF:** multiplica dos cosas:
>
> - **TF (frecuencia del término):** cuántas veces aparece la palabra en el documento.
> - **IDF (frecuencia inversa de documentos):** qué tan rara es la palabra en el corpus. Si aparece en pocos documentos, su IDF es alto.

scikit-learn calcula el IDF así, donde **n** es el número de documentos y **df** el número de documentos que contienen la palabra:

$$\text{IDF} = \ln\left(\frac{1 + n}{1 + df}\right) + 1$$

En nuestro mini corpus hay **n = 3** documentos:

| Palabra | Aparece en (df) | IDF |
|---|---|---|
| gol | 1 documento | ln(4/2) + 1 = 1,693 |
| partido | 2 documentos | ln(4/3) + 1 = 1,288 |
| tenis, tasa, interés | 1 documento cada una | 1,693 |

Para el documento 1 ("gol partido gol"):

- `gol`: TF = 2, entonces TF-IDF = 2 × 1,693 = **3,386**
- `partido`: TF = 1, entonces TF-IDF = 1 × 1,288 = **1,288**

"Partido" pesa menos porque aparece en dos documentos: es menos característica. Lo comprobamos con scikit-learn (con `norm=None` para ver los valores sin normalizar):

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_sin_norma = TfidfVectorizer(norm=None)
valores = tfidf_sin_norma.fit_transform(mini_corpus)
print("IDF:", dict(zip(tfidf_sin_norma.get_feature_names_out(), tfidf_sin_norma.idf_.round(3))))
pd.DataFrame(valores.toarray().round(3), columns=tfidf_sin_norma.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"])

**Qué deberías ver:** exactamente los valores que calculamos a mano.

**El último detalle: la normalización.** Por defecto, scikit-learn divide cada fila por su largo (norma L2), para que los documentos largos no tengan ventaja solo por tener más palabras. Después de normalizar, cada fila tiene largo 1.

In [ ]:
tfidf = TfidfVectorizer()
pd.DataFrame(tfidf.fit_transform(mini_corpus).toarray().round(3), columns=tfidf.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"])

### Ejercicio 8
Agregamos un cuarto documento: `"partido gol tenis"`. Ahora hay n = 4 documentos y "partido" aparece en 3. Calcula **a mano** el nuevo IDF de "partido" con la fórmula y guárdalo en `idf_partido`. Usa `np.log` para el logaritmo natural.

In [ ]:
idf_partido = None  # TU CÓDIGO AQUÍ
print(idf_partido)

In [ ]:
_v = TfidfVectorizer().fit(mini_corpus + ["partido gol tenis"])
revisar(lambda: np.isclose(idf_partido, _v.idf_[list(_v.get_feature_names_out()).index("partido")]))

---
# Parte 6. Puente: agrupar noticias con TF-IDF y K-medias

Juntamos las dos partes de la clase. Con TF-IDF, cada noticia es un vector de números, igual que cada país era un vector de indicadores. Entonces podemos aplicar K-medias exactamente igual que en la Parte 1.

**El corpus:** 40 noticias **ficticias**, escritas para esta clase, de 4 temas: deportes, economía, tecnología y salud. Conocemos el tema real de cada una, pero el algoritmo **no** lo verá: solo lo usaremos al final para comparar.

## Paso 6.1. Cargar y preprocesar el corpus

In [ ]:
NOTICIAS = [
    ('La selección chilena de fútbol prepara el partido clave de las clasificatorias. El equipo entrenó en el Estadio Nacional y el técnico confía en sumar puntos.', 'Deportes'),
    ('Colo-Colo y la Universidad de Chile empatan en un clásico intenso. Los dos equipos marcaron dos goles y el partido terminó con el estadio lleno.', 'Deportes'),
    ('Tenista chilena avanza a cuartos de final del torneo. Ganó el partido en tres sets y ahora enfrentará a la número uno del campeonato.', 'Deportes'),
    ('El técnico de la selección anuncia la nómina de jugadores. El equipo jugará dos partidos y busca clasificar al mundial de fútbol.', 'Deportes'),
    ('Ciclistas compiten en la vuelta por las rutas del sur. El campeonato reunió a equipos de todo el país y el ganador celebró en la meta.', 'Deportes'),
    ('El equipo de básquetbol femenino gana el campeonato sudamericano. Las jugadoras celebraron el título tras un partido muy disputado.', 'Deportes'),
    ('Futbolista chileno es fichado por un club europeo. El jugador marcó un gol en su primer partido y el equipo ganó el torneo local.', 'Deportes'),
    ('Atletas chilenos clasifican a la final de los Juegos Panamericanos. El equipo de atletismo logró su mejor marca y sueña con una medalla.', 'Deportes'),
    ('La hinchada llena el estadio para la final del campeonato. Los jugadores agradecieron el apoyo después del partido y del gol del triunfo.', 'Deportes'),
    ('La maratón de Santiago reúne a miles de corredores. Atletas de varios equipos compitieron y el ganador marcó un récord en la carrera.', 'Deportes'),
    ('El Banco Central mantiene la tasa de interés. La institución proyecta menor inflación y un crecimiento moderado de la economía el próximo año.', 'Economía'),
    ('El precio del cobre sube en los mercados. El alza fortalece las exportaciones y mejora las cuentas de la economía chilena.', 'Economía'),
    ('La inflación del mes supera lo esperado. El alza de precios en alimentos y transporte presiona el bolsillo de las familias y la economía.', 'Economía'),
    ('El desempleo baja en la región metropolitana. La encuesta de empleo muestra más trabajo en comercio y construcción, según el informe económico.', 'Economía'),
    ('El dólar cae frente al peso chileno. Los mercados reaccionaron al anuncio del Banco Central sobre la tasa de interés y la inflación.', 'Economía'),
    ('Pymes acceden a créditos con menor tasa de interés. Los bancos ofrecen préstamos para invertir y la economía local espera más empleo.', 'Economía'),
    ('El presupuesto fiscal prioriza la inversión pública. El gobierno busca impulsar el empleo y controlar el gasto para cuidar la economía.', 'Economía'),
    ('Las ventas del comercio crecen durante el mes. El aumento del consumo y la baja de la inflación ayudan a las empresas y a la economía.', 'Economía'),
    ('Exportadores de fruta reportan alza en los precios. Los envíos a Asia crecen y las exportaciones agrícolas apoyan el crecimiento económico.', 'Economía'),
    ('La bolsa de Santiago cierra al alza. Las acciones de empresas mineras subieron por el precio del cobre y los mercados celebran.', 'Economía'),
    ('Nueva aplicación con inteligencia artificial detecta fraudes en compras en línea. El sistema analiza datos de millones de transacciones digitales.', 'Tecnología'),
    ('Startup chilena lanza un robot con visión por computador. La tecnología usa inteligencia artificial y datos para clasificar fruta en la planta.', 'Tecnología'),
    ('El país amplía la cobertura de internet de fibra óptica. La conexión digital llegará a zonas rurales y mejorará el acceso a la tecnología.', 'Tecnología'),
    ('Expertos advierten sobre ciberataques a empresas. Recomiendan proteger los datos, cambiar contraseñas y actualizar el software.', 'Tecnología'),
    ('Universidad desarrolla un modelo de inteligencia artificial que entiende el español de Chile. El algoritmo aprendió de millones de datos de texto.', 'Tecnología'),
    ('Lanzan un nuevo teléfono inteligente con mejor cámara. La tecnología incluye funciones de inteligencia artificial y una batería que dura más.', 'Tecnología'),
    ('Programa de becas forma a mujeres en programación. Las participantes aprenden desarrollo de software, datos e inteligencia artificial.', 'Tecnología'),
    ('Empresas adoptan la nube y la inteligencia artificial. La tecnología digital permite automatizar procesos y analizar datos en tiempo real.', 'Tecnología'),
    ('La red 5G llega a nuevas ciudades. La tecnología promete internet más rápido y una mejor conexión para aplicaciones digitales.', 'Tecnología'),
    ('Investigadores crean un algoritmo que predice el consumo de energía. El software usa datos de sensores e inteligencia artificial.', 'Tecnología'),
    ('El ministerio de salud inicia la campaña de vacunación contra la influenza. Los centros de salud vacunarán a pacientes mayores y a niños.', 'Salud'),
    ('Hospitales reportan aumento de consultas por enfermedades respiratorias. Los médicos piden a los pacientes vacunarse durante el invierno.', 'Salud'),
    ('Estudio muestra que caminar treinta minutos al día reduce el riesgo cardiovascular. Los médicos recomiendan actividad física para cuidar la salud.', 'Salud'),
    ('Médicos recomiendan controles preventivos para detectar la diabetes. El diagnóstico temprano mejora la salud de los pacientes.', 'Salud'),
    ('Nuevo centro de salud familiar atenderá a miles de pacientes. El recinto contará con médicos, enfermeras y atención de urgencia.', 'Salud'),
    ('Especialistas advierten sobre el aumento de casos de salud mental en jóvenes. Los médicos piden más atención y apoyo a los pacientes.', 'Salud'),
    ('La lista de espera para cirugías baja en hospitales públicos. El ministerio de salud destaca la atención a pacientes con enfermedades graves.', 'Salud'),
    ('Campaña promueve la alimentación saludable en los colegios. Médicos y nutricionistas enseñan hábitos para cuidar la salud de los niños.', 'Salud'),
    ('Enfermeras y médicos reciben capacitación para atender a pacientes mayores. El hospital busca mejorar la atención de salud.', 'Salud'),
    ('Vacuna contra el virus respiratorio protege a recién nacidos en invierno. El ministerio de salud llama a vacunar en los centros de salud.', 'Salud'),
]

noticias = pd.DataFrame(NOTICIAS, columns=["texto", "tema"])
noticias["procesado"] = noticias["texto"].apply(preprocesar)
print("Noticias:", len(noticias))
noticias[["texto", "procesado"]].head(3)

## Paso 6.2. TF-IDF y K-medias

In [ ]:
vectorizador = TfidfVectorizer()
X_texto = vectorizador.fit_transform(noticias["procesado"])
print("Matriz documento-término:", X_texto.shape, "(noticias × palabras distintas)")

km_texto = KMeans(n_clusters=4, n_init=20, random_state=42).fit(X_texto)
noticias["grupo"] = km_texto.labels_
pd.crosstab(noticias["grupo"], noticias["tema"])

**Qué deberías ver:** una matriz de 40 filas y unas 300 columnas, y una tabla donde cada grupo concentra casi por completo a un tema. Quizá una noticia queda en otro grupo.

## Paso 6.3. ¿De qué habla cada grupo?

En texto, la mejor forma de interpretar un grupo es mirar las **palabras con más peso en su centroide**.

In [ ]:
terminos = vectorizador.get_feature_names_out()
for g, centroide in enumerate(km_texto.cluster_centers_):
    top = terminos[np.argsort(centroide)[::-1][:7]]
    print(f"Grupo {g}: {', '.join(top)}")

**Qué deberías ver:** grupos con palabras como `part, equip, jugador` (deportes), `econom, preci, inflacion` (economía), `inteligent, dat, artificial` (tecnología) y `salud, pacient, medic` (salud). Se ven cortadas porque usamos stemming.

## Paso 6.4. ¿Sirvió el preprocesamiento?

Comparamos con el índice de Rand ajustado (Clase 8), agrupando el texto **sin preprocesar** y **preprocesado**.

In [ ]:
from sklearn.metrics import adjusted_rand_score

X_crudo = TfidfVectorizer().fit_transform(noticias["texto"])
grupos_crudo = KMeans(n_clusters=4, n_init=20, random_state=42).fit_predict(X_crudo)

print(f"ARI sin preprocesar: {adjusted_rand_score(noticias['tema'], grupos_crudo):.3f}")
print(f"ARI preprocesado:    {adjusted_rand_score(noticias['tema'], noticias['grupo']):.3f}")

**Qué deberías ver:** el texto preprocesado se agrupa mejor. Sin preprocesar, palabras como "el", "de" y "la" acercan textos que no tienen nada que ver.

**Una advertencia:** si calculan la silueta de estos grupos, saldrá muy baja (cerca de 0,05), aunque los grupos sean buenos. Con texto hay cientos de dimensiones y casi todas valen cero en cada documento, y la silueta pierde utilidad. En texto, la mejor validación es leer las palabras clave y algunos ejemplos de cada grupo.

## Paso 6.5. Visualizar

In [ ]:
from sklearn.decomposition import TruncatedSVD

Z_texto = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_texto)
plt.figure(figsize=(8, 5))
sns.scatterplot(x=Z_texto[:, 0], y=Z_texto[:, 1], hue=noticias["tema"], style=noticias["grupo"].astype(str), s=80)
plt.title("Noticias en 2D (color: tema real, forma: grupo encontrado)"); plt.xlabel("Componente 1"); plt.ylabel("Componente 2")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.show()

Usamos `TruncatedSVD` en vez de PCA porque funciona directamente con matrices dispersas como la de TF-IDF. La idea es la misma: resumir en 2 componentes para mirar.

---
# Cierre

| Parte | Lo esencial |
|---|---|
| No supervisado | El algoritmo encuentra grupos; las personas los interpretan y deciden. La silueta orienta, pero el tamaño y el sentido de los grupos también importan. |
| PLN | El texto se obtiene, se limpia, se preprocesa y se convierte en números antes de aplicar un modelo. |
| Regex | Patrones para buscar, extraer o reemplazar texto con forma conocida. |
| Preprocesamiento | Minúsculas, quitar puntuación y stopwords, y stemming o lematización. |
| TF-IDF | Premia las palabras frecuentes en un documento pero raras en el corpus. |

**Tarea:** "Agrupa tus propios textos". Las instrucciones están en la plataforma. El plazo es el jueves 5 de noviembre a las 23:59.

Guarden el notebook en `ejercicios_estudiantes` como `clase10_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1

In [ ]:
esperanza_chile = paises.loc[paises["country"] == "Chile", "life_expec"].item()
print(esperanza_chile)

### Ejercicio 2

In [ ]:
n_comp_80 = int(np.argmax(var.cumsum() >= 0.8) + 1)
print(n_comp_80, "componentes explican", round(var.cumsum()[n_comp_80 - 1], 3))

### Ejercicio 3

In [ ]:
grupo_chile = paises.loc[paises["country"] == "Chile", "grupo"].item()
n_grupo_chile = (paises["grupo"] == grupo_chile).sum()
print(n_grupo_chile)

### Ejercicio 4
Con un eps más chico, el radio de vecindario es menor y más países quedan sin suficientes vecinos: se marcan más atípicos.

In [ ]:
n_atipicos_2 = (DBSCAN(eps=2.0, min_samples=5).fit_predict(X) == -1).sum()
print(n_atipicos_2)

### Ejercicio 5

In [ ]:
top5_ingreso = (paises[paises["grupo_nombre"] == "Necesita ayuda"]
                .sort_values("income")["country"].head(5).tolist())
print(top5_ingreso)

### Ejercicio 6
`\d+` uno o más dígitos; `(?:,\d+)?` opcionalmente una coma con dígitos; `%` el signo de porcentaje.

In [ ]:
patron_porcentaje = r"\d+(?:,\d+)?%"
print(re.findall(patron_porcentaje, texto_porcentajes))

### Ejercicio 7

In [ ]:
frase_procesada = preprocesar(otra_frase)
print(frase_procesada, "|", len(frase_procesada.split()), "tokens")

### Ejercicio 8
Con n = 4 y df = 3: IDF = ln(5/4) + 1.

In [ ]:
idf_partido = np.log((1 + 4) / (1 + 3)) + 1
print(round(idf_partido, 3))